In [14]:
import os
import numpy as np
import pandas as pd
import lightgbm as lgb

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

# CONFIG 
BASE_PATH = "/kaggle/input/mallorn-dataset"
RANDOM_STATE = 42
TIME_COL = "Time (MJD)"

# LOAD DATA
def load_lightcurves(kind):
    dfs = []
    for i in range(1, 21):
        path = f"{BASE_PATH}/split_{i:02d}/{kind}_full_lightcurves.csv"
        if os.path.exists(path):
            dfs.append(pd.read_csv(path))
    return pd.concat(dfs, ignore_index=True)

train_lc = load_lightcurves("train")
test_lc  = load_lightcurves("test")

# FEATURE ENGINEERING 
def make_features(df):
    df = df.copy()

    flux = df["Flux"].values
    mask = np.isfinite(flux)
    flux_clean = np.zeros_like(flux)
    flux_clean[mask] = np.sign(flux[mask]) * np.log1p(np.abs(flux[mask]))
    df["Flux"] = flux_clean

    df = df.sort_values(["object_id", TIME_COL])
    g = df.groupby("object_id", sort=False)

    feat = g["Flux"].agg(["mean", "std", "min", "max", "median", "skew", "count"])
    feat["p10"] = g["Flux"].quantile(0.10)
    feat["p90"] = g["Flux"].quantile(0.90)

    feat["amp"] = feat["p90"] - feat["p10"]
    feat["amp_norm"] = feat["amp"] / (feat["median"].abs() + 1e-6)
    feat["std_norm"] = feat["std"] / (feat["median"].abs() + 1e-6)

    feat["flux_diff_mean"] = g["Flux"].apply(
        lambda x: np.mean(np.abs(np.diff(x))) if len(x) > 1 else 0
    )
    feat["flux_diff_std"] = g["Flux"].apply(
        lambda x: np.std(np.diff(x)) if len(x) > 1 else 0
    )

    feat["slope"] = g.apply(
        lambda x: np.polyfit(x[TIME_COL], x["Flux"], 1)[0]
        if len(x) > 2 else 0,
        include_groups=False
    )

    feat["mad"] = g["Flux"].apply(
        lambda x: np.median(np.abs(x - np.median(x)))
    )

    return feat.reset_index().fillna(0)

# PREPARE DATA 
X_train = make_features(train_lc)
X_test  = make_features(test_lc)

train_log = pd.read_csv(f"{BASE_PATH}/train_log.csv")
test_log  = pd.read_csv(f"{BASE_PATH}/test_log.csv")

train = X_train.merge(train_log, on="object_id", how="left")
test  = X_test.merge(test_log, on="object_id", how="left")

FEATURES = [
    "mean", "std", "min", "max", "median",
    "count", "skew", "p10", "p90",
    "amp", "amp_norm", "std_norm",
    "flux_diff_mean", "flux_diff_std",
    "slope", "mad",
    "Z", "EBV"
]

X = train[FEATURES]
y = train["target"]

le = LabelEncoder()
y_enc = le.fit_transform(y)

# TRAIN MODELS 
class_counts = pd.Series(y_enc).value_counts()
class_weights = class_counts.max() / class_counts
sample_weight = np.array([class_weights[c] for c in y_enc])

lgb_train = lgb.Dataset(X, label=y_enc, weight=sample_weight)

params = {
    "objective": "multiclass",
    "num_class": len(le.classes_),
    "learning_rate": 0.05,
    "num_leaves": 31,
    "max_depth": 6,
    "min_data_in_leaf": 30,
    "feature_fraction": 0.9,
    "bagging_fraction": 0.9,
    "bagging_freq": 5,
    "verbosity": -1
}

model_lgb = lgb.train(params, lgb_train, num_boost_round=300)

rf = RandomForestClassifier(
    n_estimators=500,
    max_depth=10,
    min_samples_leaf=5,
    class_weight="balanced",
    n_jobs=-1,
    random_state=RANDOM_STATE
)
rf.fit(X, y_enc)

scaler = StandardScaler()
X_s = scaler.fit_transform(X)

lr = LogisticRegression(
    max_iter=2000,
    class_weight="balanced",
    n_jobs=-1
)
lr.fit(X_s, y_enc)

# PREDICT & SUBMIT 
proba_lgb = model_lgb.predict(test[FEATURES])
proba_rf  = rf.predict_proba(test[FEATURES])
proba_lr  = lr.predict_proba(scaler.transform(test[FEATURES]))

avg_proba = 0.45 * proba_lgb + 0.4 * proba_rf + 0.15 * proba_lr
test_pred = np.argmax(avg_proba, axis=1)

submission = pd.DataFrame({
    "object_id": test["object_id"],
    "prediction": le.inverse_transform(test_pred)
})

submission.to_csv("submission.csv", index=False)
